# Module 1: Agentic RAG

In [1]:
import os
from dotenv import dotenv_values, load_dotenv

secrets_dir = os.path.expanduser("~/Documents/.secrets/llm-zoomcamp/")

config = {
    **dotenv_values(secrets_dir + "/.env.openai"),
}

api_key = config.get("OPENAI_API_KEY")


In [2]:
from openai import OpenAI
openai_client = OpenAI(api_key=api_key)

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )
    return response.output_text

Let's give the LLM a question about the course without any context and see what happens?

In [4]:
question = "I just discovered the course, can I still join?"
answer = llm(question)
print(answer)

Absolutely — in many cases, yes.

If the course is still open for enrollment, you can usually join even if you discovered it late. A few things to check:

- **Enrollment deadline**: Some courses allow late sign-up, others don’t.
- **Course start date**: If it hasn’t started yet, you’re likely fine.
- **Missed material**: If it already began, ask whether recordings, notes, or catch-up support are available.
- **Capacity**: If the course is full, you may need to join a waitlist.

If you want, send me the course details or the message you’d like to reply with, and I can help you draft a quick inquiry.


Now lets provide the llm with some context through a more defined prompt to improve the response. I.e. make it more helpful

In [5]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [6]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [7]:
question = "I just discovered the course, can I still join?"
answer = llm(prompt)
answer

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

## Using RAG

RAG allows us to provide context to the llm to get better answers.

1. Search knowledge base for related context
2. Use context from search to enhance the main prompt
3. Provide that prompt with context to the llm

In [8]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

## Accessing the FAQ Data



In [9]:
import requests

In [10]:
docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [12]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [13]:
documents[5]

{'id': 'b71fb3b195',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: how many Zoomcamps run in a year?',
 'answer': 'DataTalks.Club runs several Zoomcamps every year. The roster and approximate timing:\n\n- Data Engineering: Jan – Apr\n- Stock Market Analytics: Apr – May\n- MLOps: May – Aug\n- LLM: Jun – Sep\n- Machine Learning: Sep – Jan\n\nFor the up-to-date list and current dates, see the [DataTalks.Club guide to free courses](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\nEach Zoomcamp has one "live" cohort per year — that\'s the only window in which you can earn the certificate. Outside the live cohort you can still take the course self-paced (materials stay open), but no certificate.'}

## Search

We are going to use minsearch to search the documents that we have gotten from the FAQ site, so we can provide the right context to the model.

In [14]:
from minsearch import Index

In [15]:
index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [16]:
index.search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

We can also filter the search results by adding a `filter_dict` parameter.

In [17]:
search_results = index.search(
    question, 
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

We can give more importance to one field over another by utilsing the `boost_dict` parameter.

In [18]:
search_results = index.search(
    question, 
    boost_dict={'question':2.0},
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

In [19]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2.0}
    filter_dict={'course': course}

    return index.search(
    question, 
    boost_dict=boost_dict,
    filter_dict=filter_dict, 
    num_results=5
)

In [20]:
search(question)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

## Building a Prompt

In [21]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [22]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [23]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: '+ doc['question'])
        lines.append('A: '+ doc['answer'])
        lines.append('')
    
    return '\n'.join(lines).strip()


In [24]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced m

In [25]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

'\nQuestion:\nI just discovered the course, can I still join?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: I missed the first homework - can I still get a certificate?\nA: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nG

In [26]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [27]:
prompt = build_prompt(question, search_results)

In [28]:
print(prompt)

Question:
I just discovered the course, can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related

## The LLM

This section covers how we can interact with the LLM api to provide the context and instructions to the model. We take a look at how the data is stored in the response object and what data is there (it is not just the response!). Then we complete the final rag pipeline using all of the other components that we established earlier.

In [29]:
response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=prompt
    )


In [30]:
response.output_text


'Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open, and complete the course with the live cohort.'

In [31]:
response.output

[ResponseOutputMessage(id='msg_08fbd2b636f7fbce006a2a51b9f7e8819ea8c4f899b27de8e5', content=[ResponseOutputText(annotations=[], text='Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open, and complete the course with the live cohort.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

By dumping the JSON from the response we can see all of the meta data that comes with the response. This includes information about how many tokens were used for the repsonse both input and output. If we were using cached inputs this would also be captured.

In [32]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_08fbd2b636f7fbce006a2a51b93afc819ea9afbe9feb88564e",
  "created_at": 1781158329.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_08fbd2b636f7fbce006a2a51b9f7e8819ea8c4f899b27de8e5",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open, and complete the course with the live cohort.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1781158330.0,
  "conversation": null,
  "ma

In [33]:
response.output[0].content[0].text

'Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open, and complete the course with the live cohort.'

In [34]:
response.usage

ResponseUsage(input_tokens=334, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=41, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=375)

We can pull specifically the usage of tokens from the object, which we can inturn use to calculate how much we are spending for each query. This will become more important later.

In [35]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00043500000000000006

When we provide the context to the model we separate this from the instructions that are essentially the system prompt of what we want the model to do, and then the user prompt includes the actual user question plus the context we get from searching the knowledge base (faq documents). This is all providied as the message history list.

In [36]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=message_history
    )

In [37]:
response.output_text

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

In [38]:
def llm_with_instructions(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
            model=model,
            input=message_history
        )
    
    return response.output_text

In [39]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm_with_instructions(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
answer = rag(question)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [41]:
rag("How do I get a certificate?")

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'